# Four kinds of posterior, one contract — and a refusal instead of a `nan`

By the time an estimand is being realized, the thing holding the draws could be a NumPyro
fit, a Laplace approximation, an analysis reloaded from disk, or a dict someone built by
hand for a test. If every function above `infer` has to know which, the package grows a
backend switch in fifty places, and the fiftieth one is wrong.

Worse is the second failure. Somebody asks a producer for a *marginal* effect and the
producer only supports counterfactuals. What comes back? In most codebases: a number. It is
computed from whatever was available, it is plausible, and nothing anywhere says it answers
a different question than the one asked.

Upper layers here are written against small protocols instead:

* `SupportsPosterior` — anything with draws you can ask questions of.
* `SupportsIntervention` — anything you can ask a counterfactual of.
* `Capability` — what a producer can do; a request needing more returns a typed
  `Unsupported` rather than a wrong number.

`Posterior` is the sampler-free implementation of `SupportsPosterior`: a dict of arrays with
an npz round-trip.

In [ ]:
import numpy as np

from axiom.core import (
    Capability,
    D,
    Intervention,
    Posterior,
    PredictiveDraws,
    SupportsEstimands,
    SupportsIntervention,
    SupportsPosterior,
    TimeWindow,
    Treatment,
    Unsupported,
    missing_capabilities,
)

from axiom.display import enable

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, density, heat, mark_x

enable();  # every axiom result renders itself from here on

## A hand-built posterior

Draws are `(chain, draw, *shape)`. `coords` label the trailing axes; `provenance` is
free-form metadata — always record the seed (the seed contract, note 0002.8). A posterior
whose seed was not recorded is a result nobody can reproduce, including the person who
produced it.

In [ ]:
from axiom.display import show

rng = np.random.default_rng(42)
post = Posterior(
    {
        "beta": rng.normal(0.3, 0.1, size=(4, 500, 2)),
        "sigma": rng.lognormal(-1, 0.2, size=(4, 500)),
    },
    coords={"treatment": ["fertilizer", "irrigation"]},
    provenance={"seed": 42, "backend": "hand-built"},
)
show(post)
print(isinstance(post, SupportsPosterior), post.names(), post.n_chains, post.n_draws())

In [ ]:
print(post.draws("beta").shape, "->", post.flat("beta").shape)
print(post.coords())
print(post.summary("sigma", definition="hdi", mass=0.9))

Every summary carries its interval definition and mass — there is no way to get a number out
of a `Posterior` without them.

In [ ]:
s = post.summary("sigma", definition="eti", mass=0.5)
print(s.interval.definition, s.interval.mass, s.interval)

In [ ]:
labels = post.coords()["treatment"]
beta = post.flat("beta")
fig = density(
    {name: beta[:, i] for i, name in enumerate(labels)},
    title="What a posterior actually is",
    subtitle="two thousand draws per treatment — the object every layer above infer reads",
    x_title="effect per unit dose",
)
mark_x(fig, 0.0, text="no effect")
caption(fig, "Coordinates carry the names, so the second axis of a (4, 500, 2) array is "
             "'fertilizer and irrigation' rather than 'column 0 and column 1' — which is the "
             "difference between a labelled result and a comment.")

## npz round-trip (no pickle)

`to_npz` writes the arrays plus a JSON metadata entry; `from_npz` loads with
`allow_pickle=False`. Gate 5 asserts no pickle anywhere in `axiom.io`: a saved posterior is
data, and loading data must not be able to run code.

In [ ]:
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
f = post.to_npz(tmp / "posterior.npz")
again = Posterior.from_npz(f)
print(again == post, again.provenance)
print(post.with_provenance(note="re-read").provenance)

## Interventions and capabilities

`SupportsIntervention` is the seam for counterfactuals. A producer declares its
`capabilities()`; before asking for something, check `missing_capabilities` and degrade with
a typed `Unsupported` instead of guessing.

In [ ]:
class ToyLinearSurface:
    """outcome = sum_k beta_k * dose_k, posterior over beta. Supports counterfactuals only."""

    def __init__(self, posterior: Posterior, treatments: list[Treatment]) -> None:
        self._post = posterior
        self._treatments = treatments

    @property
    def treatments(self) -> list[Treatment]:
        return self._treatments

    def capabilities(self) -> frozenset[Capability]:
        return frozenset({Capability.COUNTERFACTUAL})

    def predict_under(self, iv: Intervention, window: TimeWindow | None = None, seed: int | None = None) -> PredictiveDraws:
        beta = self._post.draws("beta")                       # (chain, draw, 2)
        dose = np.array([iv.doses.get(t.name, 0.0) for t in self._treatments])
        return PredictiveDraws(values=beta @ dose, intervention=iv, window=window, seed=seed)


surface = ToyLinearSurface(post, [Treatment(name="fertilizer", dimension=D.currency, unit="USD"),
                                  Treatment(name="irrigation", dimension=D.currency, unit="USD")])
print(isinstance(surface, SupportsIntervention))

In [ ]:
iv = Intervention(doses={"fertilizer": 100.0, "irrigation": 20.0}, mode="set", version="granular-v2")
pred = surface.predict_under(iv, seed=0)
print(pred.values.shape, pred.intervention.treatments, pred.intervention.version)

In [ ]:
needed = {Capability.COUNTERFACTUAL, Capability.MARGINAL}
missing = missing_capabilities(surface, needed)
if missing:
    result = Unsupported(reason=f"surface lacks {', '.join(missing)}", missing=missing)
    print(result.status, "|", result.reason, "|", bool(result))

### The capability table, drawn

Three producers a downstream function might be handed, against the six things it might ask
for. **The white cells are the whole point**: each one is a question that gets a typed
refusal naming what is missing, rather than a number computed from something else.

In [ ]:
producers = {
    "hand-built Posterior": frozenset(),
    "ToyLinearSurface": surface.capabilities(),
    "a fitted surface": frozenset(Capability),
}
caps = [c.value for c in Capability]
grid = [[float(Capability(c) in supported) for c in caps] for supported in producers.values()]

fig = heat(
    grid, caps, list(producers),
    text_fmt="{:.0f}",
    colorbar_title="supported",
    title="What each producer will answer",
    subtitle="asked for something outside its row, a producer returns Unsupported naming the gap",
    height=300,
)
caption(fig, "A blank cell is a refusal, not a zero. The alternative — every producer "
             "answering every question somehow — is how an estimand ends up reporting a "
             "counterfactual contrast under the name of a marginal effect.")

That `Unsupported` is what an estimand realization returns when the surface cannot support
it (gate 7). It is falsy, it carries a reason, and it serializes — it is never `nan` and
never an exception swallowed somewhere upstream.

`SupportsEstimands` is the full producer contract — a posterior, counterfactuals, the
declared estimands and the units it was fit in. `surface.fit` returns one; a stored analysis
or a test double can be another.

In [ ]:
print([m for m in dir(SupportsEstimands) if not m.startswith("_")])
print(isinstance(surface, SupportsEstimands), "<- the toy has no posterior, so it is not a full producer")

## What this bought you

Everything above `infer` reads draws through one interface, so a Laplace approximation, a
NUTS fit, and a hand-built dict are interchangeable in tests and in production. And the one
thing a producer cannot do is quietly answer a question it does not support.

`nbs/infer/01-backends.ipynb` puts three real samplers behind this contract;
`nbs/estimands/03-realization.ipynb` is where a missing capability becomes a refusal a
reader can see.